# Обучение модели Restormer для восстановления изображений с эффектом пикселизации
Этот Notebook демонстрирует fine-tuning Restormer (RealDenoising weights) на датасете pixelation.
Цель — PSNR 26–30 dB. Используем PaddedResize 1448×1024 + gradient checkpointing.

In [1]:
import os
import subprocess
import torch
from torch.amp import autocast, GradScaler

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.check_output(["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"]).decode().strip()
gpu_list = [tuple(map(int, line.split(', '))) for line in result.splitlines()]
best_idx, max_free = max(gpu_list, key=lambda x: x[1])
os.environ['CUDA_VISIBLE_DEVICES'] = str(best_idx)
print(f"Выбрана GPU {best_idx} — свободно ≈{max_free//1024} GiB")

device = torch.device('cuda:0')
torch.cuda.set_device(device)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()

Выбрана GPU 1 — свободно ≈46 GiB


In [7]:
# ====================== ЗАГРУЗКА МОДЕЛИ Restormer (рабочий импорт) ======================
import sys
import os

# Прямой путь к папке с restormer_arch.py — то, что у тебя сработало
sys.path.insert(0, '/home/user-4/Restormer/basicsr/models/archs')

from restormer_arch import Restormer

model = Restormer(
    inp_channels=3,
    out_channels=3,
    dim=48,
    num_blocks=[4, 6, 6, 8],
    num_refinement_blocks=4,
    heads=[1, 2, 4, 8],
    ffn_expansion_factor=2.66,
    bias=False,
    LayerNorm_type='BiasFree',
    dual_pixel_task=False,
    use_checkpoint=True
).to(device)

# Загрузка RealDenoising-весов
weights_path = '/home/user-4/pretrained/Restormer_RealDenoising.pth'
if os.path.exists(weights_path):
    checkpoint = torch.load(weights_path, map_location=device)
    model.load_state_dict(checkpoint.get('params', checkpoint), strict=False)
    print("✅ Restormer RealDenoising weights загружены успешно")
else:
    print("⚠️ Веса не найдены. Выполни wget ниже")

print(f"Модель Restormer успешно загружена. VRAM: {torch.cuda.memory_allocated(device)/1024**3:.1f} GiB")

✅ Restormer RealDenoising weights загружены успешно
Модель Restormer успешно загружена. VRAM: 0.2 GiB


In [8]:
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class PaddedResize:
    def __init__(self, size=(1448, 1024), div_factor=8):
        self.size = size
        self.div_factor = div_factor
    def __call__(self, img):
        img = transforms.Resize(self.size, interpolation=Image.LANCZOS)(img)
        w, h = img.size
        pad_w = (self.div_factor - w % self.div_factor) % self.div_factor
        pad_h = (self.div_factor - h % self.div_factor) % self.div_factor
        if pad_w or pad_h:
            img = transforms.Pad((0, 0, pad_w, pad_h), fill=0)(img)
        return img

transform = transforms.Compose([PaddedResize(), transforms.ToTensor()])

class PixelationDataset(Dataset):
    def __init__(self, distorted_dir, clean_dir, transform=None):
        self.distorted_images = sorted(os.listdir(distorted_dir))
        self.clean_images = sorted(os.listdir(clean_dir))
        self.distorted_dir = distorted_dir
        self.clean_dir = clean_dir
        self.transform = transform
    def __len__(self): return len(self.distorted_images)
    def __getitem__(self, idx):
        dist = Image.open(os.path.join(self.distorted_dir, self.distorted_images[idx])).convert('RGB')
        clean = Image.open(os.path.join(self.clean_dir, self.clean_images[idx])).convert('RGB')
        if self.transform:
            dist = self.transform(dist)
            clean = self.transform(clean)
        return dist, clean

train_dataset = PixelationDataset('pixelation/train/distorted', 'pixelation/train/clean', transform)
val_dataset   = PixelationDataset('pixelation/val/distorted',   'pixelation/val/clean',   transform)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False, num_workers=8, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 1600, Val: 400


In [9]:
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from tqdm import tqdm
import time

def charbonnier_loss(pred, target, eps=1e-6):
    return torch.mean(torch.sqrt((pred - target) ** 2 + eps))

criterion = charbonnier_loss
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
scaler = GradScaler()

patience = 12
early_stop_counter = 0
best_val_psnr = -float('inf')
os.makedirs('checkpoints', exist_ok=True)

train_losses, val_losses = [], []
train_psnrs, val_psnrs = [], []
train_ssims, val_ssims = [], []
num_epochs = 80
accum_steps = 8

In [10]:
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    train_loss = train_psnr = train_ssim = 0.0
    optimizer.zero_grad()
    step = 0

    for dist, clean in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{num_epochs} [train]"):
        dist = dist.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)
        with autocast(device_type='cuda'):
            pred = model(dist)
            loss = criterion(pred, clean) / accum_steps
        scaler.scale(loss).backward()
        step += 1
        if step % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step = 0
        train_loss += loss.item() * accum_steps
        with torch.no_grad():
            train_psnr += psnr_metric(pred, clean).item()
            train_ssim += ssim_metric(pred, clean).item()

    avg_train_loss = train_loss / len(train_loader)
    avg_train_psnr = train_psnr / len(train_loader)
    avg_train_ssim = train_ssim / len(train_loader)
    train_losses.append(avg_train_loss)
    train_psnrs.append(avg_train_psnr)
    train_ssims.append(avg_train_ssim)

    scheduler.step()

    # Валидация
    model.eval()
    val_loss = val_psnr = val_ssim = 0.0
    with torch.no_grad():
        for dist, clean in tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{num_epochs} [val]"):
            dist = dist.to(device)
            clean = clean.to(device)
            with autocast(device_type='cuda'):
                pred = model(dist)
                val_loss += criterion(pred, clean).item()
                val_psnr += psnr_metric(pred, clean).item()
                val_ssim += ssim_metric(pred, clean).item()

    avg_val_loss = val_loss / len(val_loader)
    avg_val_psnr = val_psnr / len(val_loader)
    avg_val_ssim = val_ssim / len(val_loader)
    val_losses.append(avg_val_loss)
    val_psnrs.append(avg_val_psnr)
    val_ssims.append(avg_val_ssim)

    print(f"Эпоха {epoch+1:02d} | Val PSNR: {avg_val_psnr:.2f} dB | Val SSIM: {avg_val_ssim:.4f} | Время: {time.time()-start_time:.1f}с")

    if avg_val_psnr > best_val_psnr:
        best_val_psnr = avg_val_psnr
        torch.save(model.state_dict(), "checkpoints/restormer_pixelation_best.pth")
        print("    → Лучшая модель сохранена!")
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early stopping")
            break
    torch.cuda.empty_cache()

Epoch 01/80 [train]:   0%|          | 0/800 [00:00<?, ?it/s]/home/user-4/env/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 01/80 [train]:   6%|▋         | 50/800 [07:19<1:49:46,  8.78s/it] 


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'finetuned_restormer_pixelation.pth')
print("Финальная модель сохранена как finetuned_restormer_pixelation.pth")

import matplotlib.pyplot as plt
fig, axs = plt.subplots(3, 1, figsize=(12, 18))
axs[0].plot(train_losses, label='Train Loss')
axs[0].plot(val_losses, label='Val Loss')
axs[0].set_title('Loss')
axs[1].plot(train_psnrs, label='Train PSNR')
axs[1].plot(val_psnrs, label='Val PSNR')
axs[1].set_title('PSNR')
axs[2].plot(train_ssims, label='Train SSIM')
axs[2].plot(val_ssims, label='Val SSIM')
axs[2].set_title('SSIM')
plt.tight_layout()
plt.savefig('restormer_pixelation_metrics.png')
plt.show()
print("Графики сохранены: restormer_pixelation_metrics.png")